# C/GMRES

C/GMRESは Continuation / GMRES という非線形制御問題の制約条件を考慮した制御入力をPMP条件とGMRESにより求めるものである。

PMP条件を以下に示す。Hamilltonianを用いると表記が簡単になるが、まずは理解のためこのように記述する。また、最初であるため、時間 $t$ も記述する。

状態方程式

$$
\boxed{
\dot{x}(t)= f(x(t), u(t), t)
}
$$

随伴方程式

$$
\boxed{
\dot {\lambda}(t) = - L_x(x(t),u(t), t) - {f_x(x(t),u(t), t)}^T \lambda(t)
}
$$

停留条件
$$
\boxed{
L_u(x(t),u(t), t) + {f_u(x(t),u(t), t)}^T \lambda(t) = 0
}
$$

終端条件

$$
\boxed{
\lambda(T) = {\Phi_x}(x(T),T)
}
$$


## PMP条件をC/GMRESの予測ホライゾンで求める方法

PMPは以下の最適化制御問題を上記の4つの方程式に変換して、時間 $[0,T]$ すべてにおいて、これらを満たす $x(t), u(t), \lambda(t)$ を求める問題である。 

$$
\min\limits_{x(t), u(t)} \space J = \Phi(x(T), T) + \int_0^T L (x,u,t) dt \\
\text{subject to} \space \dot{x} = f(x,u,t), \space x(0) = x_0
$$

ここでは有限の時間区間 $[0,T]$ における最適制御問題にPMPを適用している。
一方、実際の制御ではずっと制御ループを回し続けるため、実時間 $t$ が進み続ける。<br>
C/GMRESでは以下の図のように、各時刻 $t_k$ において、現在時刻から一定時間先までの予測ホライゾンを設定する。<br>
予測ホライゾン内の時間を予測時間 $\tau$ として、 $\tau \in [0,T]$ においてPMP条件を適用し、最適な制御入力の時系列を求める。

<img src="images/c-gmres_time_images.png" style="width:60%;">

実時間 $t$ と予測時間 $\tau$ は異なり、$\tau$ 方向の計算中に実時間 $t$ が進んでいることを意味するものではない。 

PMPの計算は以下のように進めていく。状態 $x$ と随伴変数 $\lambda$ は多変数のベクトルも考慮できるが、ここでは理解のため $x$, $\lambda$ は1変数とする。



### Step 1

制御入力 $U(t_k)$ を 予測ホライゾンの時間区間 $[0, T]$ で離散時間で設定する。

$$
U(t_k) = [u_0(t_k), u_1(t_k), \cdots, u_{N-1}(t_k)] \\
$$

C/GMRESの初期では $U(0)$ を設定するが、2回目以降は前回の計算結果をそのまま用いる。

### Step 2

状態方程式 $\dot{x}(\tau)= f(x(\tau), u(\tau), t_k + \tau)$ をオイラー法で時間区間 $[0,T]$ まで初期状態 $x[0] = x(t_k)$ から $x[N]$ の時系列を求める。$h$ はステップ幅である。

$$
\boxed{
x[n+1] = x[n] + h \ f(x[n], u_n(t_k), t_k + \tau[n]) , \space n = 0,1, \cdots , N-1
}
$$

ここで求まる $x$ の予測ホライゾン上の時系列は以下のものである。

$$
x[0], x[1] , \cdots , x[N]
$$

### Step 3

終端条件から、随伴変数の最終値 $\lambda[N]$ をStep1で求めた状態 $x[N]$ を用いて求める。

$$
\boxed{
\lambda[N] = {\Phi_x}(x[N], t_k + T)
}
$$

### Step 4

随伴方程式 $\dot {\lambda}(\tau) = - L_x(x(\tau),u(\tau), t_k + \tau) - {f_x(x(\tau),u(\tau), t_k + \tau)}^T \lambda(\tau)$ を時間を逆方向に進める $N-1$ から $1$ まで計算して、随伴変数 $\lambda$ の予測ホライゾン上の時系列を求める。$x$は1変数であるため本来は $f_x$ の転置は必要ないが、記述として残している。

$$
\boxed{
\lambda[n] = \lambda[n+1] + h L_x(x[n], u_n(t_k), t_k + \tau[n]) + \left( h f_x(x[n], u_n(t_k), t_k + \tau[n]) \right)^T\lambda[n+1]  , \quad n = N-1 , N-2, \cdots , 1
}
$$

この計算式は以下の考え方に基づいている。

----------------------------------------------------------------------------

評価関数 $J$ を離散化する。積分区間は区分求積法としている。

$$
J = \Phi(x[N]) + \sum_{n=0}^{N-1} \ h \ L(x[n], u_n, t[n])
$$

制約条件はオイラー法を用いた離散化により、以下のようになる。

$$
x[n+1] - x[n] - h \ f(x[n], u_n, t[n]) = 0
$$

PMPの拡大評価関数に合わせて制約条件を組み込むと、離散化した拡大評価関数 $\bar{J}$ は以下のようになる。

$$
\bar{J} = \Phi(x[N]) + \sum_{n=0}^{N-1} \left( \ h \ L(x[n], u_n, t[n]) + \lambda[n+1]^T(x[n] + h \ f(x[n], u_n, t[n]) - x[n+1] ) \right)
$$

PMPの随伴方程式は 拡大評価関数の $x$ による偏微分から $\dot {\lambda} = - L_x(x,u) - {f_x(x,u)}^T \lambda$ のように求めたため、 上記の $\bar{J}$ を $L$と $f$ の引数である $x[n]$ で偏微分する。

$$
\frac{\partial \bar{J}}{\partial x[n]} = 0
$$

積分内部をいくつか展開する。

$$
\begin{aligned}
h L (x[0], u_0, t[0]) + \lambda[1]^T(x[0] + h f(x[0],u_0, t[0]) - x[1]) \quad , n=0 \\
h L (x[1], u_1, t[1]) + \lambda[2]^T(x[1] + h f(x[1],u_1, t[1]) - x[2]) \quad , n=1 \\
h L (x[2], u_2, t[2]) + \lambda[3]^T(x[2] + h f(x[2],u_2, t[2]) - x[3]) \quad , n=2 \\
\end{aligned}
$$

これを $x[n]$ についてまとめると、以下のように $\lambda[n]^T x[n]$ が一つずつ移っていく。 

$$
\begin{array}{l}
h L (x[0], u_0, t[0]) + \lambda[1]^T (x[0] + h f(x[0], u_0, t[0])) + \\
h L (x[1], u_1, t[1]) + \lambda[2]^T (x[1] + h f(x[1], u_1, t[1])) - \lambda[1]^T \ x[1] + \\
h L (x[2], u_2, t[2]) + \lambda[3]^T (x[2] + h f(x[2], u_2, t[2])) - \lambda[2]^T \ x[2] + \\
\vdots \\
h L (x[N-1], u_{N-1}, t[N-1]) + \lambda[N]^T (x[N-1] + h f(x[N-1], u_{N-1}, t[N-1])) - \lambda[N-1]^T \ x[N-1]
\end{array}
$$

よって、$ 1 \le n \le N-1$ の区間では以下のようになる。

$$
h L (x[n], u_n, t[n]) + \lambda[n+1]^T (x[n] + h f(x[n], u_n, t[n])) - \lambda[n]^T \ x[n]
$$

これを $x[n]$ で変微分し "ゼロ" とする。

$$
h L_x(x[n], u_n, t[n]) + \lambda[n+1] + \left( h f_x(x[n], u_n, t[n]) \right)^T \lambda[n+1]  - \lambda[n] = 0
$$

ここから、$\lambda$ の 時系列を次のように計算できる。

$$
\lambda[n] = \lambda[n+1] + h L_x(x[n], u_n, t[n]) + \left( h f_x(x[n], u_n, t[n]) \right)^T\lambda[n+1]  , \quad n = N-1 , N-2, \cdots , 1
$$

----------------------------------------------------------------------------

これにより、次の $\lambda$ の時系列が求まる。

$$
\lambda[N], \lambda[N-1] , \cdots , \lambda[1]
$$

### Step 5

制御入力 $U(t_k)$ を仮定し、 Step 1から Step 4 で 状態変数 $x[n]$ 、随伴変数 $\lambda[n]$、の時系列を予測ホライゾン区間を求めた。<br>
Step 4 ではこれらを使い、停留条件の式 $L_u(x(\tau),u(\tau), t_k + \tau) + {f_u(x(\tau),u(\tau), t_k + \tau)}^T \lambda(\tau) = 0$ を 具体的な数値の縦のベクトルを形成する。これを $F$ として定義する。

$$
\boxed{
F(U(t_k), x(t_k), t_k) =
\begin{bmatrix}
L_u(x[0],u_0(t_k), t_k+\tau[0]) + {f_u(x[0],u_0(t_k), t_k + \tau[0])}^T \lambda[1] \\
L_u(x[1],u_1(t_k), t_k+\tau[1]) + {f_u(x[1],u_1(t_k), t_k + \tau[1])}^T \lambda[2] \\
\vdots \\
L_u(x[N-1],u_{N-1}(t_k), t_k+\tau[N-1]) + {f_u(x[N-1],u_{N-1}(t_k), t_k + \tau[N-1])}^T \lambda[N]
\end{bmatrix} = 0
}
$$

この $F(U(t_k), x(t_k), t_k) = 0 $ は$t_k$から始まる予測ホライゾン上のすべての離散点で、停留条件を満たすことを意味している。



### 次の制御入力 $U(t_{k+1})$の計算

Step 5 では $t_k$ における制御入力 $U(t_k)$から状態変数 $x[n]$ 、随伴変数 $\lambda[n]$ の時系列を求め、予測ホライゾン上の停留条件をまとめた $F(U(t_k), x(t_k), t_k)$ を求めた。<br>
理想的には $F=0$ を満たす制御入力を次の時刻 $t_{k+1}$ においても追従したい。<br>
まずは理想的に $F=0$が成立しているとして、その状態を次の時刻でも維持するために、時間変化がゼロであるとして次の関係を導く。

$$
\frac{d \ F ( U(t_k), x(t_k), t_k)}{d \ t} = 0
$$

上式は $t$ による全微分であるため、チェインルールを適用すると以下となる。記述の簡略化のため、$t_k=t$とする。

$$
\begin{aligned}
\frac{d \ F ( U(t), x(t), t)}{d \ t} &= \frac{\partial \ F }{\partial \ U} \frac{ d U}{dt} + \frac{\partial \ F }{\partial \ x} \frac{ d x}{dt} + \frac{\partial \ F }{\partial \ t} \\
&= F_U \dot{U} + F_x \dot{x} + F_t = 0
\end{aligned}
$$

- $F_U \ \dot{U}$ : $U$ が実時間方向に変化することによる $F$ の変化率
- $F_x \ \dot{x}$ : $x$ が実時間方向に変化することによる $F$ の変化率
- $F_t$ : 予測ホライゾンの起点 $t_k$ が進むことによる、実時間への明示的既存を通した $F$ の変化率

これより、以下の式が現れる。

$$
F_U \dot{U} = -\left(F_x \dot{x} + F_t \right)
$$

$U$はベクトルであるため、その偏微分 $F_U$ はヤコビアン(行列)になる。また、右辺はベクトルになるため、この式は $A z = b$ のように、一次連立方程式であり、GMRESで解くことが出来る。<br>
そして、これを解くことにより、$\dot{U}$が求まるため、次の制御入力は以下のように求めることが出来る。

$$
\boxed{
U(t_{k+1}) = U(t_k) + \Delta t \ \dot{U}(t_k)
}
$$

ここまでで停留条件を満たしつつ、制御入力 $U$ を計算していく方法を説明した。


### 初期制御入力 $U(0)$ の決定

ここまでの説明では、初期時刻 $t_k=0$ において、 $F(U(0),x(0),0)=0$ を満たす制御入力 $U(0)$ が与えられており、最初から停留条件を満たした状態で制御されるとして扱った。

しかし、実際には制御開始時にこの制御入力列を決める必要がある。C/GMRESでは、予測ホライゾンの長さ$T$ を可変として、制御ループの初期は $T(0)=0$ とすることで初期化問題を単純化する。

上記では状態、随伴変数の時系列を求めて $F$を構成した。このとき、時系列は予測ホライゾン上の時間長$T$をN個で分割した$h$を用いた。<br>
ここでは $h = T(0)/N=0$となるため、予測ホライゾン上の状態は

$$
x[0] = x[1] = \cdots = x[N] = x(0)
$$

となり、終端条件と随伴変数は以下となる。

$$
\lambda[1] = \cdots = \lambda[N] = \Phi_x(x(0), 0)
$$

この時各予測点で同じ制御入力 $u(0)$ を用いると停留条件はすべて同一の以下の代数方程式となる。

$$
L_u(x(0), u(0), 0) + f_u (x(0), u(0), 0)^T \Phi_x(x(0),0)= 0
$$

上記代数方程式は、$u(0)$ が未知数であり、この$u(0)$ を解析的または数値的に求めて、

$$
U(0) = \left[ u(0), u(0), \cdots , u(0)\right]^T
$$

とすることで、

$$
F(U(0), x(0), 0)  = 0
$$

を満たす制御入力列を構成できる。

$$
F_U(0)\dot{U}(0) = -(F_x(0) \dot{x}(0) + F_t(0))
$$

を $\dot{U}(0)$ について解き、次の時刻 の制御入力 $U(t_1)$ を求めれば、停留条件 $F=0$ を時間方向へ追従することができる制御入力となっている。

$$
U(t_1) = U(0) + \Delta t \ \dot{U}(0)
$$

初期の$U(0)$は 予測ホライゾンの長さを $T(0)=0$ として計算したが、以下のように時間とともにその長さを最終長$T_f$まで伸ばしていく。

$$
T(t) = T_f(1 - \exp(-\alpha t))
$$

そして、PMPで時系列を計算するときのステップ幅は経過時間とともに以下で計算する。

$$
h(t_k)=T(t_k)/N
$$  

#### 解析的に $u(0)$ 解く例

モデルを質点とおいて、状態を$\mathbf{x}(t) = [x(t), v(t)]^T$とすると以下の運動方程式になる。

$$
\begin{aligned}
f(t) &= m \frac{d^2 x(t)}{dt^2} \\
v(t) &= \frac{d x(t)}{d t}
\end{aligned} \rightarrow
\begin{aligned}
\frac{d x(t)}{d t} &= v(t) \\
\frac{d v(t)}{d t} &= \frac{1}{m} f(t)
\end{aligned} \rightarrow
\frac{d}{d t}
\begin{bmatrix}
x(t) \\ v(t)
\end{bmatrix} =
\begin{bmatrix}
0 & 1 \\
0 & 0 
\end{bmatrix}
\begin{bmatrix}
x(t) \\ v(t)
\end{bmatrix}
+ \begin{bmatrix}
 0 \\ 1/m
\end{bmatrix} f(t)
$$

これより、状態方程式 $\dot{\mathbf{x}}(t) = f(\mathbf{x}(t), u(t))$は以下となる。

$$
\dot{\mathbf{x}}(t) = A \mathbf{x}(t) + B u(t), \quad u(t) =f(t)\quad A = \begin{bmatrix}
0 & 1 \\
0 & 0 
\end{bmatrix}, \quad B = \begin{bmatrix}
 0 \\ 1/m
\end{bmatrix} 
$$

ランニングコスト $L(\mathbf{x}(t),u(t))$を次のように設定する。$Q,R$は対称行列とする。

$$
L(\mathbf{x}(t), u(t)) = \frac{1}{2}(\mathbf{x}(t) - x_{ref})^T \ Q \ (\mathbf{x}(t) - x_{ref}) + \frac{1}{2} u(t)^T \ R \ u(t)
$$

ここで$u(t)$ はスカラであるため、$R$もスカラである。

終端コスト $\Phi (\mathbf{x}(T))$ を次のように設定する。$Q_T = Q_T^T$ の対称行列とする。

$$
\Phi (\mathbf{x}(T)) = \frac{1}{2}(\mathbf{x}(T) - x_{ref})^T Q_T (\mathbf{x}(T) - x_{ref})
$$

これから停留条件を求めるため、$f_u, \Phi_x, L_u$ を計算する。

$$
\begin{aligned}
f_u &= B \\
\Phi_x &= Q_T (\mathbf{x}(T) - x_{ref}) , \quad \because \ \Phi_x \equiv \frac{\partial \Phi(\mathbf{x}(T))}{\partial \mathbf{x}(T)}\\
L_u &= R u(t)
\end{aligned}
$$


$T(0)=0$ とすると、停留条件 $L_u + {f_u}^T \lambda = 0$ は $\lambda(0) = \Phi_x(0)$ となるため

$$
R u(0) + B^T Q_T(\mathbf{x}(0) - x_{ref}) = 0
$$

ここから、$u(0)$ を以下のように計算できる。

$$
u(0) = - \frac{B^T Q_T(\mathbf{x}(0) - x_{ref})}{R}
$$

#### 数値的に $u(0)$ を解く例

数値計算で初期 $u(0)$ を解く方法として、Newton-Raphson法がある。

ランニングコストを入力コストをできるだけ小さくしたいというようにし次のものとする。

$$
L(\mathbf{x}(t), u(t)) = \frac{1}{2}(\mathbf{x} - x_{ref})^T Q (\mathbf{x} - x_{ref}) + \frac{1}{2}R u(t)^2 + \frac{1}{4}S u(t)^4 , \quad R > 0, S > 0
$$

状態方程式は上記の質点モデルを用いる。また終端条件も上記と同一とする。

$$
\dot{\mathbf{x}}(t) = A \mathbf{x}(t) + B u(t)
$$


停留条件に必要な $f_u, \Phi_x, L_u$ は以下となる。

$$
\begin{aligned}
f_u &= B \\
\Phi_x &= Q_T (\mathbf{x}(T) - x_{ref}) \\
L_u &= R u(t) + S u(t)^3
\end{aligned}
$$

したがって、$T(0) = 0$ における停留条件 $L_u + f_u^T \lambda = 0$ は

$$
G(u) = Ru(0) + S u(0)^3 + B^T Q_T (\mathbf{x}(0) - x_{ref}) = 0
$$

となり、$u$に対して非線形となるため Newton-Raphson法により数値的に $u(0)$ を探す。

$G(u)$を$u(0)$で偏微分すると

$$
G_u(u) =\frac{\partial G}{\partial u}= R + 3S u^2
$$

これより、適当な$u(0)$の初期値を$u^{(0)}(0)$ とすると、以下の繰り返し計算で$u(0)$ の近似値を求めることが出来る。

$$
u^{(j+1)}(0) = u^{(j)}(0) - \frac{G(u^{(j)}(0))}{G_u(u^{(j)}(0))} = u^{(j)}(0) - \frac{R u^{(j)}(0) + S u^{(j)}(0)^3 + B^T Q_T (\mathbf{x}(0) - x_{ref})}{R + 3 S u^{(j)}(0)^2}
$$

以下の$\Delta u$ が十分に小さいときに、$u(0) = u^{(j+1)}(0)$ として計算を終了する。

$$ 
\Delta u = |u^{(j+1)}(0) - u^{(j)}(0)|
$$


### $\dot{U}$ の計算

以下の式に含まれる $F_U, F_x , F_t$ を解析的に求めることもできるが、計算が膨大になる。そこで C/GMRESでは有限差分で右辺と左辺のそれぞれを求めていく。

$$
F_U \dot{U} = -\left(F_x \dot{x} + F_t \right)
$$

まず右辺の計算方法を考える。

#### 右辺を構成する $F_x \dot{x} + F_t$ 

$U$ は固定して、$x,t$ の少し先を以下のようにする。

$$
\Delta x = \varepsilon \dot{x} , \quad \Delta t = \varepsilon
$$

すると、$F(U, x+\Delta x , t + \Delta t)$ の多変数のテイラー展開を1次変化までとすると、以下のようになる。

$$
F(U, x+\Delta x , t + \Delta t) \simeq F(U, x, t) + F_x \Delta x + F_t \Delta t
$$

ここで、$\Delta x = \varepsilon \dot{x} , \ \Delta t = \varepsilon$ を代入すると以下のようになる。

$$
\begin{aligned}
F(U, x+ \varepsilon \dot{x}, t + \varepsilon) &\simeq F(U, x, t) + F_x \varepsilon \dot{x}  + F_t \varepsilon \\
F(U, x+ \varepsilon \dot{x}, t + \varepsilon) &\simeq F(U, x, t) + \varepsilon  \left(F_x \dot{x}  + F_t \right) \\
F(U, x+ \varepsilon \dot{x}, t + \varepsilon) - F(U, x, t) &\simeq \varepsilon  \left(F_x \dot{x}  + F_t \right) \\
\end{aligned}
$$

これより、以下の有限差分で近似できる。

$$
F_x \dot{x}  + F_t \simeq \frac{F(U, x+ \varepsilon \dot{x}, t + \varepsilon) - F(U, x, t) }{\varepsilon} \\
$$

C/GMRESの記述に合わせると、以下のようになる。

$$
\boxed{
F_x \dot{x}  + F_t \simeq \frac{F(U(t_k), x(t_k)+ \varepsilon \dot{x}(t_k), t_k + \varepsilon) - F(U(t_k), x(t_k), t_k) }{\varepsilon} \\
}
$$

これは、$F_x \dot{x}  + F_t$ を求めるために、Step1 から Step 5を $x$ と $t$ に関するものを設定しなおして $F$を再計算する。

有限差分により、$$x(t_k) + \varepsilon \dot{x}(t_k)$と$t_k + \varepsilon$は一括して計算されるため、どちらの影響があったのかは計算結果からは判断ができない。しかし、どのような項目が影響するのかを確認のため考える。

##### $x(t_k) + \varepsilon \dot{x}(t_k)$ の影響

Step2 では 初期値 $x[0] = x(t_k)$として 状態の時系列を求めたが、これを $x_\varepsilon[0] = x(t_k) + \varepsilon \dot{x}(t_k)$ として計算していく。ここで $x(t_k)$ は実時間が $t_k$ 時の制御対象の状態である。

これにより、$x_\varepsilon[0], x_\varepsilon[1], \cdots , x_\varepsilon[N]$ の時系列をStep2で再計算する。

ここで、$\dot{x}(t_k)$ は以下のように状態方程式から求めることが出来る。

$$
\dot{x}(t_k) = f(x(t_k), u_0(t_k) , t_k)
$$

例えば制御対象が簡単に質点モデルである場合、状態を$[x(t_k), v(t_k)]$とすると、状態方程式は以下となる。

$$
\frac{d}{dt}
\begin{bmatrix}
x(t_k) \\ v(t_k)
\end{bmatrix} =
\begin{bmatrix}
v(t_k) \\ u_0(t_k)/m
\end{bmatrix}
$$

よって、$t_k$時刻の $v(t_k)$ と制御入力 $u_0(t_k)$があれば計算できる。 状態はすべて観測可能と考えてC/GMRESの計算を行うが、観測できなればKalman Filter等の状態推定を行う必要がある。しかし、これはC/GMRESとは異なるトピックであるため説明は省く。また、$u_0(t_k)$は予測ホライゾン上の最初の制御入力の値である。

##### $t_k + \varepsilon$ の影響

$t_k + \varepsilon$ は $F$ に含まれる実時間 $t$ に明示的に依存する項を、微小時間 $\varepsilon$ だけ進めて評価することを意味する。

例えば目標値 $x_{ref}(t)$ が時間変化する場合、ランニングコスト $L$ を考えた場合、連続時間では以下となる。

$$
L(x,u,t) = \frac{1}{2} q ( x - x_{ref}(t))^2 + \frac{1}{2} r \ u^2
$$ 

ここで目標値 を $x_{ref}(t) = \sin(t)$ とした場合、予測ホライゾン上の$n$番目の時刻は $t = t_k + \tau[n]$ になるため、$L$は以下となる。

$$
L(x[n] , u_n, t_k + \tau[n]) =  \frac{1}{2} q ( x[n] - \sin(t_k + \tau[n]))^2 + \frac{1}{2} r \ u_n^2
$$

これを $\varepsilon$ 進めて考えるため以下のようになる。

$$
L(x_\varepsilon[n], u_n, t_k + \varepsilon + \tau[n]) = \frac{1}{2} q ( x_\varepsilon[n] - \sin(t_k + \varepsilon + \tau[n]))^2 + \frac{1}{2} r \ u_n^2
$$

これが $t_k + \varepsilon$ による $F_t$ に現れる一つの影響である。<br>
これ以外にも終端コストや状態方程式でも明示的に実時間 $t$ に依存する項目などが考えられる。

#### 左辺を構成する $F_U \dot{U}$

式を再掲する。

$$
F_U \dot{U} = -\left(F_x \dot{x} + F_t \right)
$$

ここで GMRESの手法を思い出すと、

GMRESは $A x = b$ の一次連立方程式を初期値 $x_0$ を設定し、初期残差$r_0 = b - A x_0$ からKrylov部分空間の基底ベクトル $v$ をArnoldi法で求めて、繰り返し計算で残差を小さくし、最終的にkrylov部分空間内の最小化問題から $y_m$計算して、$x = x_0 + V_m y_m$ と近似解を求める手法であった。

このとき $v_1 = r_0 / ||r_0||$ と求まり、この$v_1$ をから、次のStepでは $A v_1$ でKrylov部分空間を広げていく。
また、次のStep でも 一つ前で求めた $v_2$ を使い、$A v_2$ Krylov部分空間を広げることを繰り返す。

この $A x = b$ は $F_U \dot{U} = -\left(F_x \dot{x} + F_t \right)$ であるため、求めたい $\dot{U}(t_k)$ の初期値 $\dot{U}^{(0)}(t_k)$ から　$r_0 = -\left(F_x \dot{x} + F_t \right) - F_U \dot{U}^{(0)}(t_k)$ とし $v_1 = r_0 / ||r_0||$ で求めればよい。

##### $\dot{U}(t_k)$が求まっていない場合

前回の $\dot{U}(t_k)$ が存在しないため、簡単のため $\dot{U}^{(0)}(t_k) = 0$ とする。これより $F_U \dot{U}^{(0)}(t_k) = 0$ となり、 

$$
\boxed{
r_0 = -(F_x \dot{x} + F_t)
}
$$

と $r_0$ が計算できるので、

$$
v_1 = \frac{r_0}{||r_0||}
$$

と計算することが出来る。この $v_1$ から、Krylov 部分空間を広げるための $F_U v_1$ は次の有限差分で求めることが出来る。

$\Delta U = \varepsilon v$ とすると、$x,t$ を固定して $F$ の多変数テイラー 展開の1次展開までをまとめると以下となる。

$$
F(U(t_k) + \Delta U, x(t_k), t_k) \simeq F(U(t_k), x(t_k), t_k) + F_U \Delta U
$$

これに $\Delta U$ を代入してまとめると、$F_U v$ が有限差分で求まる。

$$
\begin{aligned}
F(U(t_k) + \varepsilon v, x(t_k), t_k) \simeq F(U(t_k), x(t_k), t_k) + F_U \varepsilon \ v \\
\boxed{
F_U \ v \simeq \frac{F(U(t_k) + \varepsilon v, x(t_k), t_k) - F(U(t_k), x(t_k), t_k)}{\varepsilon}\\
}
\end{aligned}
$$

ここで $v$ は任意に設定するベクトルではなく、GMRES内部のArnoldi法によって生成されるkrylov部分空間の基底ベクトルである。

制御入力が1次元の場合、

$$
U(t_k) = \begin{bmatrix}
u_0(t_k) \\ u_1 (t_k) \\ \vdots \\ u_{N-1}(t_k)
\end{bmatrix}
$$

であり、Arnoldi法で計算される $v$ も同じ次元で以下となる。

$$
v = \begin{bmatrix}
v_0 \\ v_1 \\ \vdots \\ v_{N-1}
\end{bmatrix}
$$

よって、$U(t_k) + \varepsilon v$ は以下となる。

$$
U(t_k) + \varepsilon v = \begin{bmatrix}
u_0(t_k) + \varepsilon v_0 \\ u_1(t_k) + \varepsilon v_1 \\ \vdots \\ u_{N-1}(t_k) + \varepsilon v_{N-1}
\end{bmatrix}
$$

この$U(t_k) + \varepsilon v$を用いてStep1からStep5まで計算する。このとき $\tilde{x} [0]=x(t_k)$ であり、$t_k$も変化させていない。<br>
例えば状態方程式を用いて時系列を計算する場合、

$$
\tilde{x} [0]=x(t_k)
$$

から初めて以下を計算する。

$$
\tilde{x}[n+1] = \tilde{x}[n] + h \ f(\tilde{x}[n] , u_n(t_k) + \varepsilon v_n, t_k + \tau[n])
$$

この方法で $\dot{U}(t_k) = V_m  y_m$ と計算することが出来る。

ここから、$U(t_{k+1}) = U(t_k) + \Delta t \dot{U}(t_k)$ と計算できる。

##### $\dot{U}(t_k)$が求まっている場合

GMRESで $F_U \dot{U} = -\left(F_x \dot{x} + F_t \right)$ を解き、$\dot{U}(t_k)$ を求めた後は、その次の時刻 $t_{k+1}$ のGMRESの計算で残差 $r_0$ を以下のように計算できる。

$$
\boxed{
r_0(t_{k+1}) = -\left(F_x \dot{x} + F_t \right)_{t_{k+1}} - F_U(t_{k+1}) \dot{U}(t_k)
}
$$

この $F_U(t_{k+1}) \dot{U}(t_k)$ は有限差分で次のように計算できる。

$x,t$ を固定して $U$の少し先を次のようにする。

$$
\Delta U = \varepsilon \dot{U}
$$

すると $F(U + \Delta U , x, t)$ の多変数のテイラー展開の1次変化までとすると、

$$
F(U + \Delta U , x, t) \simeq F(U , x, t) + F_U \Delta U 
$$

ここで $\Delta U$ を代入すると、

$$
\begin{aligned}
F(U + \varepsilon \dot{U}, x, t) \simeq& F(U , x, t) + F_U \ \varepsilon \dot{U} \\
F(U + \varepsilon \dot{U}, x, t) - F(U , x, t) \simeq& F_U \ \varepsilon \dot{U} \\
F_U \ \dot{U} \simeq& \frac{F(U + \varepsilon \dot{U}, x, t) - F(U , x, t)}{\varepsilon}
\end{aligned}
$$

GMRESの表記に合わせると、

$$
\boxed{
F_U(t_{k+1}) \ \dot{U}(t_k) \simeq \frac{F(U(t_{k+1}) + \varepsilon \dot{U}(t_k), x(t_{k+1}), t_{k+1}) - F(U(t_{k+1}) , x(t_{k+1}), t_{k+1})}{\varepsilon}
}
$$

これにより、$F_U \ \dot{U}$ が求まるため、$r_0$を計算し、その後のGMRESの計算を進めて $F_U \dot{U} = -\left(F_x \dot{x} + F_t \right)$の近似解を以下のように求める。

$$
\dot{U}(t_{k+1}) = \dot{U}(t_k) + V_m y_m
$$

そして、次の制御入力を以下のように求める。

$$
U(t_{k+2}) = U(t_{k+1}) + \Delta t \ \dot{U}(t_{k+1})
$$



### $F$ の安定化について

上記までは 初期の制御入力 $U(0)$ を解析的または数値的に求めて $F=0$ として 次の制御入力を求める方法を展開した。

しかし、実際には初期の制御入力や数値誤差により $F=0$ が完全には成立しない。

そこで次の安定化を行う。

$$
\frac{d F}{dt} = -\zeta F, \quad \zeta > 0
$$

この式の解は 
$$
F(t) = F(0) \exp(-\zeta t)
$$

となるため、時間の経過とともに $F$がゼロに近づいていく。

F(U, x, t) であるため、上式は以下となる。

$$
F_U \dot{U} + F_x \dot{x} + F_t = -\zeta F 
$$

これから、$\dot{U}$を求めるための一次連立方程式は以下となる。

$$
F_U \dot{U} = - (F_x \dot{x} + F_t + \zeta F) 
$$

よって、右辺が $b= -(F_x \dot{x} + F_t + \zeta F)$ となり、これを有限差分で以下のように求める。

$$
\boxed{
b = - \frac{F(U, x+ \varepsilon \dot{x}, t+ \varepsilon) - F(U, x, t)}{\varepsilon} - \zeta F(U, x, t)
}
$$ 

安定化を導入しても、有限差分による $F_U v$ の計算やGMRESのアルゴリズム自体は変わらず、変化する箇所はGMRESへ渡す右辺 $b$ のみである。


## C/GMRESの流れ

以上でC/GMRESを計算するための方法が整った。この章で流れをまとめていく。

予測ホライゾンの区間は $[0,T(t_k)]$ とする。この区間の最終時間は $T(t_k) = T_f(1-\exp(- \alpha t_k)), \ \alpha > 0$ と時変であり、$t_0=0$では $T(t_0)=0$だが、時刻が進むにつれて、$T(t_k) \rightarrow T_f$ へ近づいていく。

予測ホライゾンの区間を$N$分割し、状態 $x$ と随伴変数 $\lambda$ の時系列のデータを計算するときのステップ幅 $h$を計算する。$T(t_k)$ は時変であるため、ステップ幅も時変である。

$$
h(t_k) = \frac{T(t_k)}{N}
$$

また、予測ホライゾン上の離散時間$\tau[n]$は ステップ幅 $h$ より以下のように構成されるため、こちらも時変である。

$$
\tau[n] = h(t_k) \ n , \quad n = 0,1, \cdots , N
$$

よって、時刻 $t_k$ により $T(t_k), h(t_k), \tau[n]$ が変化する。

### 制御ループの初回 時刻 $t_0$

実時間 $t_0$では $T(t_0)=0$ である。<br>
初回の制御入力 $u(t_0)$ を計算するため、制御対象の状態 $x(t_0)$を取得する。そして解析的または数値的に$u(0)$を求め、以下のように $N$ 個の制御入力列 $U(t_0)$ を構成する。

$$
U(t_0) = [u(t_0), \cdots , u(t_0)]^T , U(t_0) \in \mathbb{R}^{N}
$$

この$u(t_0)$を制御対象へ出力する。

また、$F_x \dot{x} + F_t$ を計算するため、$\dot{x}$ を状態方程式から以下のように計算する。

$$
\dot{x}(t_0) = f(x(t_0), u(t_0), t_0)
$$

#### $F(U(t_0), x(t_0), t_0)$ を求める

状態 $x$ の時系列 $([x[0], \cdots , x[N]])$ を求める。

ここで計算に用いるステップ幅 $h(t_0) = T(t_0)/N = 0$ であるため、時系列は次のようになる。

$$
x[n+1] = x[n] = x(t_0), \quad n = 0,1, \cdots ,N-1
$$

終端条件から随伴変数 $\lambda[N]$ 求める。

$$
\lambda[N] = \Phi_x(x(t_0), t_0 + T(t_0))
$$

随伴変数 $\lambda$ の時系列 $([\lambda[N], \cdots , \lambda[1]])$ を求める。

$$
\lambda[n] = \lambda[n+1] = \Phi_x(x(t_0), t_0 + T(t_0)), \quad n = N-1, \cdots ,1
$$

求めた状態と随伴変数の時系列から $F(U(t_0), x(t_0), t_0)$ を計算する。

ここで予測ホライゾン上の時刻$\tau$は、予測ホライゾンの長さをN分割したときの1ステップだが、$T(t_0)=0$では $\tau[n]=0$ となる。

$$
F(U(t_0), x(t_0), t_0) = 
\begin{bmatrix}
L_u (x(t_0), u(t_0), t_0 + \tau[0]) + f_u(x(t_0), u(t_0), t_0 + \tau[0])^T \ \Phi_x(x(t_0), t_0 + T(t_0)) \\
\vdots \\
L_u (x(t_0), u(t_0), t_0 + \tau[N-1]) + f_u(x(t_0), u(t_0), t_0 + \tau[N-1])^T \ \Phi_x(x(t_0), t_0 + T(t_0)) \\
\end{bmatrix}
$$

$F(U(t_0), x(t_0), t_0)$ の列の要素はすべて同じだが、上記のように列を構成する。これは有限差分を計算するためである。

#### GMRES計算で必要な要素を有限差分で求める

記述の簡略化のため、$F_U(t_0)$を次のように書く。

$$
F_U(t_0) := F_U(U(t_0), x(t_0), t_0)
$$

以下の式をGMRESで解き、$\dot{U}(t_0)$ を求める。

$$
F_U(t_0) \dot{U}(t_0) = -(F_x(t_0) \dot{x}(t_0) + F_t(t_0) + \zeta F(t_0)) , \quad \zeta > 0
$$

ただし、初期化により $F(t_0)=0$ が成立している場合、初回では安定化項 $\zeta F(t_0)=0$ となる。

これを $A x = b$ の表現でとらえると、$A = F_U(t_0), \ x = \dot{U}(t_0), \ b = -(F_x(t_0) \dot{x}(t_0) + F_t(t_0) + \zeta F(t_0))$ である。

右辺 $b = -(F_x(t_0) \dot{x}(t_0) + F_t(t_0) + \zeta F(t_0))$ は一度だけの計算であり、左辺は繰り返し計算になる。

##### 右辺 $b_{t_0}$ の計算

状態 $x_\varepsilon$の時系列 $([x_\varepsilon[0], \cdots , x_\varepsilon[N]])$ を初期値 $x_\varepsilon[0] = x(t_0) + \varepsilon \dot{x}(t_0)$ から求める。

また、時刻は $t_0 + \varepsilon$ となるため区間は$T(t_0 + \varepsilon) = T_f(1 - \exp(-\alpha(t_0 + \varepsilon))) > 0 $である。よって、ステップ幅は以下となる。

$$
h_\varepsilon = h(t_0 + \varepsilon) = \frac{T(t_0 + \varepsilon)}{N}
$$

また、予測ホライゾン上の各時刻 $\tau$ は$h_\varepsilon$により以下となる。

$$
\tau_\varepsilon[n] = n h_\varepsilon, n = 0, 1, \cdots , N - 1
$$

よって状態 $x_\varepsilon$の時系列は以下となる。

$$
x_\varepsilon[n+1] = x_\varepsilon[n] + h_\varepsilon f(x_\varepsilon[n] , u(t_0), t_0 + \varepsilon + \tau_\varepsilon[n]) , \quad n = 0,1, \cdots ,N-1
$$

終端条件から随伴変数 $\lambda_\varepsilon[N]$ 求める。

$$
\lambda_\varepsilon[N] = \Phi_x(x_\varepsilon[N], t_0 + \varepsilon + T(t_0 + \varepsilon))
$$

随伴変数 $\lambda_\varepsilon$ の時系列 $([\lambda_\varepsilon[N], \cdots , \lambda_\varepsilon[1]])$ を求める。

$$
\lambda_\varepsilon[n] = \lambda_\varepsilon[n+1] + h_\varepsilon L_x( x_\varepsilon[n] , u(t_0), t_0 + \varepsilon + \tau_\varepsilon[n]) + (h_\varepsilon f_x( x_\varepsilon[n] , u(t_0), t_0 + \varepsilon + \tau_\varepsilon[n]))^T \lambda_\varepsilon[n+1], \quad n = N-1, \cdots ,1
$$

求めた状態と随伴変数の時系列から$F(U(t_0), x(t_0) + \varepsilon \dot{x}(t_0), t_0 + \varepsilon)$を計算する。

$$
F(U(t_0), x(t_0) + \varepsilon \dot{x}(t_0), t_0 + \varepsilon) = 
\begin{bmatrix}
L_u (x_\varepsilon[0], u_0(t_0), t_0 + \varepsilon + \tau_\varepsilon[0]) + f_u(x_\varepsilon[0], u_0(t_0), t_0 + \varepsilon+ \tau_\varepsilon[0])^T \ \lambda_\varepsilon[1] \\
\vdots \\
L_u (x_\varepsilon[N-1], u_{N-1}(t_0), t_0 + \varepsilon+ \tau_\varepsilon[N-1]) + f_u(x_\varepsilon[N-1], u_{N-1}(t_0), t_0 + \varepsilon+ \tau_\varepsilon[N-1])^T \ \lambda_\varepsilon[N] \\
\end{bmatrix}
$$

$F(U(t_0), x(t_0), t_0)$ とは異なり、一般には $F(U(t_0), x(t_0)+ \varepsilon \dot{x}(t_0), t_0+ \varepsilon)$ の列の要素は異なる。

$b = -(F_x(t_0) \dot{x}(t_0) + F_t(t_0) + \zeta F(t_0))$ を以下の有限差分を計算する。
実際には近似値だが、制御ループではこの近似値を用いるため、"=" で記述する。

$$
b = - \frac{F(U(t_0), x(t_0) + \varepsilon \dot{x}(t_0), t_0 + \varepsilon) - F(U(t_0), x(t_0), t_0)}{\varepsilon}  - \zeta F(U(t_0), x(t_0), t_0)
$$

$F(U(t_0), x(t_0)+ \varepsilon \dot{x}(t_0), t_0+ \varepsilon)$ のため、$b$の各要素も一般には異なる。

このように、右辺の有限差分では実時間を $t_0 + \varepsilon$ とし評価する。このとき $T(t_0 + \varepsilon) > 0$ となるため予測ホライゾンにも有限の長さが生じる。そのため状態と随伴変数の時系列が生成され、$F(U(t_0), x(t_0) + \varepsilon \dot{x}(t_0), t_0 + \varepsilon)$の各成分は一般には異なる値となる。<br>
この変化には状態 $x$ の変化による $F_x \dot{x}$ と、実時間および $T(t)$ の変化による $F_t$ の双方が含まれているが、この有限差分では両者を個別には分離しない。

##### $\dot{U}(t_0)$ の計算

GMRESの計算を行って、$F_U(t_0) \dot{U}(t_0) = b$ から、$\dot{U}(t_0)$ の近似解を求める。

##### Arnoldi法 Step 0 初期残差 $r_0$ の決定

$r_0$ の計算に用いる 解の初期値を $\dot{U}(t_0)^{(0)} = 0$ として以下のように計算する。

$$
r_0 = b - F_U(t_0) \dot{U}(t_0)^{(0)} = b 
$$

これより、Krylov部分空間の初期基底 $v_1$ は以下となる。

$$
\begin{aligned}
\beta &= ||r_0|| \\
v_1 &= r_0 / \beta 
\end{aligned}
$$

##### Arnoldi法 Step m 

Step m 回目は以下のように計算する。

$$
\begin{aligned}
F_U(t_0) v_{m} &= \frac{F(U(t_0) + \varepsilon v_m, x(t_0), t_0) - F(U(t_0), x(t_0), t_0)}{\varepsilon} \\
h_{i,m} &= {v_{i}}^T F_U(t_0) v_{m} \space (i=(1, \ 2, \ \cdots \ m)) \\
w &= F_U(t_0) v_{m} -\sum_{i=1}^m h_{i,m}v_i \\
h_{m+1, m} &= ||w|| \\
v_{m+1} &= w / h_{m+1, m}
\end{aligned}
$$

$F_U(t_0) v_{m}$ は次のように有限差分で計算を行う。

--------------------------------------------------------------
$U(t_0) + \varepsilon v_{m}$ を $U(t_0)$ をもとに以下のように構成する。

$$
U(t_0) + \varepsilon v_{m} = \begin{bmatrix}
u_0(t_0) + \varepsilon v_{m}[0] \\ u_1(t_0) + \varepsilon v_{m}[1] \\ \vdots \\ u_{N-1}(t_0) + \varepsilon v_{m}[N-1]
\end{bmatrix}
$$

$m=1$の場合、$v_1= r_0 / ||r_0|| = b / ||b||$ であり、bの要素が一般には異なるため、$U(t_0) + \varepsilon v_{m}$の要素も一般には異なる。

ここでは$U$のみを微小変化させるため、$T(t_0)=0$ ではステップ幅 $h = 0$ となる。また、予測ホライゾン上の時刻 $\tau$ も $\tau[n]=0$ である。

状態 $\tilde{x}$ の時系列 $([\tilde{x}[0], \cdots , \tilde{x}[N]])$ を初期値 $\tilde{x}[0] = x(t_0)$ から求める。

$$
\tilde{x}[n+1] = \tilde{x}[n]=x(t_0) , \quad n = 0,1, \cdots ,N-1
$$

終端条件から随伴変数 $\tilde{\lambda}[N]$ 求める。

$$
\tilde{\lambda} [N] = \Phi_x(x(t_0), t_0 + T(t_0))
$$

随伴変数 $\tilde{\lambda}$ の時系列 $([\tilde{\lambda} [N], \cdots , \tilde{\lambda} [1]])$ を求める。

$$
\tilde{\lambda} [n] = \tilde{\lambda} [n+1] = \Phi_x(x(t_0), t_0 + T(t_0)) , \quad n = N-1, \cdots ,1
$$

求めた状態と随伴変数の時系列から $F(U(t_0) + \varepsilon v_m , x(t_0), t_0)$ を計算する。

$$
F(U(t_0) + \varepsilon v_m , x(t_0), t_0) = 
\begin{bmatrix}
L_u (x(t_0), u_0(t_0) + \varepsilon v_m[0], t_0  + \tau[0]) + f_u(x(t_0), u_0(t_0) + \varepsilon v_m[0], t_0 +  \tau[0])^T \ \Phi_x(x(t_0), t_0 + T(t_0)) \\
\vdots \\
L_u (x(t_0), u_{N-1}(t_0) + \varepsilon v_m[N-1], t_0 + \tau[N-1]) + f_u(x(t_0), u_{N-1}(t_0) + \varepsilon v_m[N-1], t_0 + \tau[N-1])^T \ \Phi_x(x(t_0), t_0 + T(t_0)) \\
\end{bmatrix}
$$

$F(U(t_0) + \varepsilon v_m , x(t_0), t_0)$の要素は $u_n(t_0) + \varepsilon v_m[n] \ (n = 0, \cdots, N-1) $ が一般には異なるため、これらも一般には異なっている。

$F_U(t_0) v_m$ を以下の有限差分で計算する。

$$
F_U(t_0) v_m = \frac{F(U(t_0) + \varepsilon v_m, x(t_0), t_0) - F(U(t_0), x(t_0), t_0)}{\varepsilon}
$$

--------------------------------------------------------------

この後は、Givens回転による上ヘッセンベルグ行列の回転させ、上三角行列と右辺の $g$ を計算し、残差が小さければ最小化問題の解の $Y_m$ を後退代入で解き、Krylov部分空間の基底 $V_m$ を用いて、以下の計算で $\dot{U}(t_0)$ を求める。

$$
\dot{U}(t_0) = \dot{U}(t_0)^{(0)} + V_m Y_m = V_m Y_m
$$

そして、次の制御入力 $U(t_1)$ を以下のように求める。

$$
U(t_1) = U(t_0) + \Delta t \ \dot{U}(t_0)
$$

## 2回目以降の制御ループ 時刻 $t_k$


時刻 $t_0$ の制御ループで $U(t_1)$ と $\dot{U}(t_0)$ が求まった。そして次の時刻から予測ホライゾンの長さ $T(t_k) > 0 , k=1, \cdots $ と有限長になる。そのため、状態と随伴変数の時系列を求めるステップ幅$h(t_k)$と予測ホライゾン上の離散時間$\tau[n]$は0ではない有限値となる。 

- $T(t_k) = T_f (1 - \exp(-\alpha t_k)) , \quad \alpha >0 $ を計算する。
- $h(t_k) = T(t_k) / N$ を計算する
- $\tau[n] = h(t_k) n, n = 0, \cdots N$ とする。

制御入力 $U(t_k)$ は前回の制御ループより下記のように求められている。

$$
U(t_k) = [u_0(t_k), \cdots , u_{N-1}(t_k)]^T , U(t_k) \in \mathbb{R}^{N}
$$

状態 $x(t_k)$を制御対象の状態から取得し、$u_0(t_k)$ を制御対象に出力する。

そして、$F_x \dot{x} + F_t$ を計算するため、$\dot{x}$ を状態方程式から以下のように計算する。

$$
\dot{x}(t_k) = f(x(t_k), u_0(t_k), t_k)
$$

#### $F(U(t_k), x(t_k), t_k)$ を求める

状態 $x$ の時系列 $([x[0], \cdots , x[N]])$ を x[0] = $x(t_l)$ として求める。

$$
x[n+1] = x[n]  + h(t_k) f(x[n] , u_n(t_k), t_k + \tau[n]), \quad n = 0,1, \cdots ,N-1
$$

終端条件から随伴変数 $\lambda[N]$ 求める。

$$
\lambda[N] = \Phi_x(x[N], t_k + T(t_k))
$$

随伴変数 $\lambda$ の時系列 $([\lambda[N], \cdots , \lambda[1]])$ を求める。

$$
\lambda[n] = \lambda[n+1] + h(t_k) L_x(x[n] , u_n(t_k), t_k + \tau[n]) + (h(t_k) f_x(x[n], u_n(t_k), t_k + \tau[n]))^T \lambda[n+1], \quad n = N-1, \cdots ,1
$$

求めた状態と随伴変数の時系列から $F(U(t_k), x(t_k), t_k)$ を計算する。

$$
F(U(t_k), x(t_k), t_k) = 
\begin{bmatrix}
L_u (x[0], u_0(t_k), t_k + \tau[0]) + f_u(x[0], u_0(t_k), t_k + \tau[0])^T \ \lambda[1] \\
\vdots \\
L_u (x[N-1], u_{N-1}(t_k), t_k + \tau[N-1]) + f_u(x[N-1], u_{N-1}(t_k), t_k + \tau[N-1])^T \ \lambda[N] \\
\end{bmatrix}
$$

#### GMRES計算で必要な要素を有限差分で求める

以下の式をGMRESで解き、$\dot{U}(t_k)$ を求める。

$$
F_U(t_k) \dot{U}(t_k) = -(F_x(t_k) \dot{x}(t_k) + F_t(t_k) + \zeta F(t_k)) , \quad \zeta > 0 , \quad F_U(t_k) := F_U(U(t_k), x(t_k), t_k)
$$

これを $A x = b$ の表現でとらえると、$A = F_U(t_k), \ x = \dot{U}(t_k), \ b = -(F_x(t_k) \dot{x}(t_k) + F_t(t_k) + \zeta F(t_k))$ である。

右辺 $b = -(F_x(t_k) \dot{x}(t_k) + F_t(t_k) + \zeta F(t_k))$ は一度だけの計算であり、左辺は繰り返し計算になる。

##### 右辺 $b_{t_k}$ の計算

状態 $x_\varepsilon$の時系列 $([x_\varepsilon[0], \cdots , x_\varepsilon[N]])$ を初期値 $x_\varepsilon[0] = x(t_k) + \varepsilon \dot{x}(t_k)$ から求める。

また、時刻は $t_k + \varepsilon$ となるため区間は$T(t_k + \varepsilon) = T_f(1 - \exp(-\alpha(t_k + \varepsilon))) > 0 $である。よって、ステップ幅は以下となる。

$$
h_\varepsilon = h(t_k + \varepsilon) = \frac{T(t_k + \varepsilon)}{N}
$$

また、予測ホライゾン上の各時刻 $\tau_varepsilon$ は$h_\varepsilon$により以下となる。

$$
\tau_\varepsilon[n] = n h_\varepsilon, n = 0, 1, \cdots , N 
$$

よって状態 $x_\varepsilon$の時系列は以下となる。

$$
x_\varepsilon[n+1] = x_\varepsilon[n] + h_\varepsilon f(x_\varepsilon[n] , u_n(t_k), t_k + \varepsilon + \tau_\varepsilon[n]) , \quad n = 0,1, \cdots ,N-1
$$

終端条件から随伴変数 $\lambda_\varepsilon[N]$ 求める。

$$
\lambda_\varepsilon[N] = \Phi_x(x_\varepsilon[N], t_k + \varepsilon + T(t_k + \varepsilon))
$$

随伴変数 $\lambda_\varepsilon$ の時系列 $([\lambda_\varepsilon[N], \cdots , \lambda_\varepsilon[1]])$ を求める。

$$
\lambda_\varepsilon[n] = \lambda_\varepsilon[n+1] + h_\varepsilon L_x( x_\varepsilon[n] , u_n(t_k), t_k + \varepsilon + \tau_\varepsilon[n]) + (h_\varepsilon f_x( x_\varepsilon[n] , u_n(t_k), t_k + \varepsilon + \tau_\varepsilon[n]))^T \lambda_\varepsilon[n+1], \quad n = N-1, \cdots ,1
$$

求めた状態と随伴変数の時系列から $F(U(t_k), x(t_k) + \varepsilon \dot{x}(t_k), t_k + \varepsilon)$ を計算する。

$$
F(U(t_k), x(t_k) + \varepsilon \dot{x}(t_k), t_k + \varepsilon) = 
\begin{bmatrix}
L_u (x_\varepsilon[0], u_0(t_k), t_k + \varepsilon + \tau_\varepsilon[0]) + f_u(x_\varepsilon[0], u_0(t_k), t_k + \varepsilon+ \tau_\varepsilon[0])^T \ \lambda_\varepsilon[1] \\
\vdots \\
L_u (x_\varepsilon[N-1], u_{N-1}(t_k), t_k + \varepsilon+ \tau_\varepsilon[N-1]) + f_u(x_\varepsilon[N-1], u_{N-1}(t_k), t_k + \varepsilon+ \tau_\varepsilon[N-1])^T \ \lambda_\varepsilon[N] \\
\end{bmatrix}
$$

$b = -(F_x(t_k) \dot{x}(t_k) + F_t(t_k) + \zeta F(t_k))$ を以下の有限差分を計算する。
実際には近似値だが、制御ループではこの近似値を用いるため、"=" で記述する。

$$
b = - \frac{F(U(t_k), x(t_k) + \varepsilon \dot{x}(t_k), t_k + \varepsilon) - F(U(t_k), x(t_k), t_k)}{\varepsilon}  - \zeta F(U(t_k), x(t_k), t_k)
$$

##### $\dot{U}(t_k)$ の計算

GMRESの計算を行って、$F_U(t_k) \dot{U}(t_k) = b$ から、$\dot{U}(t_k)$ の近似解を求める。

##### Arnoldi法 Step 0 初期残差 $r_0$ の決定

$r_0$ の計算に用いる 解の初期値 $\dot{U}(t_k)^{(0)}$ を前回の制御ループで求めた$\dot{U}(t_{k-1})$ として以下のように計算できる。

$$
r_0 = b - F_U(t_k) \dot{U}(t_k)^{(0)} = b - F_U(t_k) \dot{U}(t_{k-1}) 
$$

ここで $F_U(t_k) \dot{U}(t_{k-1})$ をそのままヤコビアンとの積を計算するのではなく、Arnoldi法の$F_Uv$ と同様に以下の有限差分で求める。

---------------------------------------------------------------------
$U(t_k) + \varepsilon \dot{U}(t_{k-1})$ を $U(t_k)$ をもとに以下のように構成する。

$$
U(t_k) + \varepsilon \dot{U}(t_{k-1}) = \begin{bmatrix}
u_0(t_k) + \varepsilon \dot{u}_0(t_{k-1}) \\ u_1(t_k) + \varepsilon \dot{u}_1(t_{k-1}) \\ \vdots \\ u_{N-1}(t_k) + \varepsilon \dot{u}_{N-1}(t_{k-1})
\end{bmatrix}
$$

状態 $\hat{x}$ の時系列 $([\hat{x}[0], \cdots , \hat{x}[N]])$ を初期値 $\hat{x}[0] = x(t_k)$ から求める。

$$
\hat{x}[n+1] = \hat{x}[n] + h(t_k) f(\hat{x}[n] , u_n(t_k) + \varepsilon \dot{u}_n(t_{k-1}), t_k + \tau[n]) , \quad n = 0,1, \cdots ,N-1
$$

終端条件から随伴変数 $\hat{\lambda}[N]$ 求める。

$$
\hat{\lambda} [N] = \Phi_x(\hat{x}[N], t_k + T(t_k))
$$

随伴変数 $\hat{\lambda}$ の時系列 $([\hat{\lambda} [N], \cdots , \hat{\lambda} [1]])$ を求める。

$$
\hat{\lambda} [n] = \hat{\lambda} [n+1] + h(t_k) L_x(\hat{x}[n], u_n(t_k) + \varepsilon \dot{u}_n(t_{k-1}), t_k + \tau[n]) + (h(t_k) f_x(\hat{x}[n], u_n(t_k) + \varepsilon \dot{u}_n(t_{k-1}), t_k + \tau[n]))^T \hat{\lambda}[n+1] , \quad n = N-1, \cdots ,1
$$

求めた状態と随伴変数の時系列から $F(U(t_k) + \varepsilon  \dot{U}(t_{k-1}), x(t_k), t_k)$ を計算する。

$$
F(U(t_k) + \varepsilon \dot{U}(t_{k-1}) , x(t_k), t_k) = 
\begin{bmatrix}
L_u (\hat{x}[0], u_0(t_k) + \varepsilon \dot{u}_0(t_{k-1}), t_k  + \tau[0]) + f_u(\hat{x}[0], u_0(t_k) + \varepsilon \dot{u}_0(t_{k-1}), t_k +  \tau[0])^T \ \hat{\lambda}[1] \\
\vdots \\
L_u (\hat{x}[N-1], u_{N-1}(t_k) + \varepsilon \dot{u}_{N-1}(t_{k-1}), t_k + \tau[N-1]) + f_u(\hat{x}[N-1], u_{N-1}(t_k) + \varepsilon \dot{u}_{N-1}(t_{k-1}), t_k + \tau[N-1])^T \ \hat{\lambda}[N] \\
\end{bmatrix}
$$

$F_U(t_k) \dot{U}(t_{k-1})$ を以下の有限差分で計算する。

$$
F_U(t_k) \dot{U}(t_{k-1}) = \frac{F(U(t_k) + \varepsilon \dot{U}(t_{k-1}), x(t_k), t_k) - F(U(t_k), x(t_k), t_k)}{\varepsilon}
$$

---------------------------------------------------------------------



これより、Krylov部分空間の初期基底 $v_1$ は以下となる。

$$
\begin{aligned}
\beta &= ||r_0|| \\
v_1 &= r_0 / \beta 
\end{aligned}
$$

##### Arnoldi法 Step m 

Step m 回目は以下のように計算する。

$$
\begin{aligned}
F_U(t_k) v_{m} &= \frac{F(U(t_k) + \varepsilon v_m, x(t_k), t_k) - F(U(t_k), x(t_k), t_k)}{\varepsilon} \\
h_{i,m} &= {v_{i}}^T F_U(t_k) v_{m} \space (i=(1, \ 2, \ \cdots \ m)) \\
w &= F_U(t_k) v_{m} -\sum_{i=1}^m h_{i,m}v_i \\
h_{m+1, m} &= ||w|| \\
v_{m+1} &= w / h_{m+1, m}
\end{aligned}
$$

$F_U(t_k) v_{m}$ は次のように有限差分で計算を行う。

--------------------------------------------------------------
$U(t_k) + \varepsilon v_{m}$ を $U(t_k)$ をもとに以下のように構成する。

$$
U(t_k) + \varepsilon v_{m} = \begin{bmatrix}
u_0(t_k) + \varepsilon v_{m}[0] \\ u_1(t_k) + \varepsilon v_{m}[1] \\ \vdots \\ u_{N-1}(t_k) + \varepsilon v_{m}[N-1]
\end{bmatrix}
$$

状態 $\tilde{x}$ の時系列 $([\tilde{x}[0], \cdots , \tilde{x}[N]])$ を初期値 $\tilde{x}[0] = x(t_k)$ から求める。

$$
\tilde{x}[n+1] = \tilde{x}[n] + h(t_k) f(\tilde{x}[n] , u_n(t_k) + \varepsilon v_m[n], t_k + \tau[n]) , \quad n = 0,1, \cdots ,N-1
$$

終端条件から随伴変数 $\tilde{\lambda}[N]$ 求める。

$$
\tilde{\lambda} [N] = \Phi_x(\tilde{x}[N], t_k + T(t_k))
$$

随伴変数 $\tilde{\lambda}$ の時系列 $([\tilde{\lambda} [N], \cdots , \tilde{\lambda} [1]])$ を求める。

$$
\tilde{\lambda} [n] = \tilde{\lambda} [n+1] + h(t_k) L_x(\tilde{x}[n], u_n(t_k) + \varepsilon v_m[n], t_k + \tau[n]) + (h(t_k) f_x(\tilde{x}[n], u_n(t_k) + \varepsilon v_m[n], t_k + \tau[n]))^T \tilde{\lambda}[n+1] , \quad n = N-1, \cdots ,1
$$

求めた状態と随伴変数の時系列から $F(U(t_k) + \varepsilon v_m , x(t_k), t_k)$ を計算する。

$$
F(U(t_k) + \varepsilon v_m , x(t_k), t_k) = 
\begin{bmatrix}
L_u (\tilde{x}[0], u_0(t_k) + \varepsilon v_m[0], t_k  + \tau[0]) + f_u(\tilde{x}[0], u_0(t_k) + \varepsilon v_m[0], t_k +  \tau[0])^T \ \tilde{\lambda}[1] \\
\vdots \\
L_u (\tilde{x}[N-1], u_{N-1}(t_k) + \varepsilon v_m[N-1], t_k + \tau[N-1]) + f_u(\tilde{x}[N-1], u_{N-1}(t_k) + \varepsilon v_m[N-1], t_k + \tau[N-1])^T \ \tilde{\lambda}[N] \\
\end{bmatrix}
$$

$F_U(t_k) v_m$ を以下の有限差分で計算する。

$$
F_U(t_k) v_m = \frac{F(U(t_k) + \varepsilon v_m, x(t_k), t_k) - F(U(t_k), x(t_k), t_k)}{\varepsilon}
$$

--------------------------------------------------------------

この後は、Givens回転による上ヘッセンベルグ行列の回転させ、上三角行列と右辺の $g$ を計算し、残差が小さければ最小化問題の解の $Y_m$ を後退代入で解き、Krylov部分空間の基底 $V_m$ を用いて、以下の計算で $\dot{U}(t_k)$ を求める。

$$
\dot{U}(t_k) = \dot{U}(t_k)^{(0)} + V_m Y_m
$$

そして、次の制御入力 $U(t_{k+1})$ を以下のように求める。

$$
U(t_{k+1}) = U(t_k) + \Delta t \ \dot{U}(t_k)
$$

今回求めた$U(t_{k+1})$の第1要素 $u_0(t_{k+1})$は、次のC/GMRES制御ループの先頭で状態 $x(t_{k+1})$ を取得した後、制御対象へ出力される。

##### $t_0$ と $t_k$ 時の処理の差

1. $U(t_0)$は解析的または数値的に求めるが、$U(t_k), k \ge 1$ は前回の制御ループで更新された値を用いる。
1. $t_0$では予測ホライゾンの時間ステップ$h$が右辺を計算する時を除き $h=0$ となり、状態と随伴変数は同じ値となる。
1. $t_0$ではGMRESで$\dot{U}(t_0)$を求めるときの初期解$\dot{U}^{(0)}(t_0)=0$となるが、$t_k$では前回の制御ループで求めた$\dot{U}(t_{k-1})$を以下のように初期解として用いて、warm startする。そして、$F_U(t_k) \dot{U}(t_{k-1})$ を有限差分で求める。 
$$
\dot{U}^{(0)}(t_k)=\dot{U}(t_{k-1})
$$

以上以外は計算手順は同一である。<br>
初期の制御ループでは$T(t_0) = 0$ という初期条件のため式の記述が$t_k$の時と異なるが、実装上は同じ関数群を$t_k$と$T(t_k)$を与えて使用できるようにしておくとよい。

### まとめ

以上によりC/GMRESのアルゴリズムの内容を説明した。

実時間 $t_k$ におけるC/GMRESは以下のような見通しになる。

$$
\boxed{
\begin{array}{c}
x(t_k) を取得 \space (t_0ではu(t_0)をもとめてU(t_0)を構成)\\ \downarrow \\
u_0(t_k) を出力 \\ \downarrow \\
F(U(t_k), x(t_k), t_k)を計算 \\ \downarrow \\
b_k を有限差分で計算 \\ \downarrow \\
\dot{U}^{(0)}(t_k) = \dot{U}(t_{k-1}) \space (t_0では、\dot{U}^{(0)}(t_0) = 0) \\ \downarrow \\
r_0 = b_k - F_U(t_k)\dot{U}^{(0)}(t_k) \space (F_U(t_k)\dot{U}{(0)}(t_k)は有限差分で計算) \\ \downarrow \\
\text{GMRES}で \dot{U}(t_k)=\dot{U}^{(0)}(t_k) + V_m Y_m を計算 \ (F_U v_m は有限差分で計算) \\ \downarrow \\
U(t_{k+1}) = U(t_k) + \Delta t \dot{U}(t_k)
\end{array}
}
$$

今回のC/GMRESは等式制約を対象としたC/GMRESについて説明した。不等式制約を扱う場合、最適性条件 $F(U,x,t) = 0$ の構成を拡張する必要があるが、Continuation により $F=0$ を追跡し、その仮定で生じる線形方程式GMRESで得意という基本的な計算の流れは同じである。

以上